# 🔑 Day 1 — OpenAI API Key & 첫 번째 API 호출

---

## 🎯 오늘의 상황

> AI 서비스를 직접 만들어 보고 싶은데, 어디서부터 시작해야 할까요?  
> 가장 먼저 해야 할 일은 **OpenAI에 "나 여기 있어요!"** 라고 알리는 것입니다.  
> 그게 바로 **API Key** 입니다.
>
> 오늘은 API Key를 발급받고, **GPT에게 처음으로 말을 걸어봅니다.**

---

## 📋 오늘 배울 것

```
STEP 1. API Key란 무엇인가? (개념)
STEP 2. API Key 안전하게 저장하기 (Colab Secrets)
STEP 3. OpenAI 클라이언트 만들기
STEP 4. 첫 번째 API 호출 해보기
STEP 5. 응답 구조 파헤치기
STEP 6. 비용 계산해보기
```

---

## 💡 핵심 개념 미리 보기

| 용어 | 한 줄 설명 |
|------|------------|
| **API Key** | GPT에게 "나 유효한 사용자야"를 증명하는 비밀 열쇠 |
| **Colab Secrets** | API Key를 코드 밖에 안전하게 보관하는 금고 |
| **클라이언트** | GPT와 대화하기 위한 연결 통로 |
| **토큰** | GPT가 텍스트를 읽는 최소 단위 (과금 기준) |
| **finish_reason** | GPT가 왜 대답을 멈췄는지 알려주는 신호 |

---
## STEP 1. API Key란 무엇인가?

### 🏨 호텔 룸 카드키로 이해하기

```
API Key  =  호텔 룸 카드키

카드키가 있어야 → 문이 열린다     (API 호출 가능)
카드키를 잃으면 → 타인이 들어온다 (누군가 내 계정으로 과금)
분실하면        → 새 카드키 발급  (Key 폐기 후 재발급)
```

### 📌 API Key 기본 정보

| 항목 | 내용 |
|------|------|
| 생긴 모양 | `sk-` 로 시작하는 긴 문자열 |
| 발급 위치 | platform.openai.com → API Keys |
| 주의 사항 | 발급 직후 **딱 한 번만** 볼 수 있음 → 즉시 복사! |
| 유출 시 위험 | 타인이 내 계정으로 API 사용 → 요금 폭탄 |

> ⚠️ **Key를 코드에 직접 쓰면 안 되는 이유**  
> 노트북을 공유하거나 GitHub에 올리는 순간 Key도 같이 공개됩니다!  
> → **Colab Secrets** 에 저장하는 것이 정답입니다.

---
## STEP 2. API Key 안전하게 저장하기 (Colab Secrets)

### 📦 저장 방법 비교

| 방법 | 예시 | 위험도 |
|------|------|--------|
| 코드에 직접 작성 | `api_key="sk-abc123..."` | ❌ 절대 금지 |
| **Colab Secrets** | `userdata.get("OPENAI_API_KEY")` | ✅ 안전 |

### 🔧 Colab Secrets 설정 방법 (4단계)

```
① 왼쪽 사이드바 → 🔑 아이콘 클릭
② + 새 보안 비밀 추가 클릭
③ 이름: OPENAI_API_KEY  /  값: 발급받은 Key 전체 붙여넣기
④ 노트북 액세스 토글 → 파란색(활성화) ← 이 단계를 빠뜨리면 작동 안 함!
```

---
## STEP 3. OpenAI 클라이언트 만들기

클라이언트는 **GPT와 대화하기 위한 전용 창구**입니다.  
한 번 만들어두면 이후 모든 대화는 이 창구를 통해 이루어집니다.

In [1]:
from google.colab import userdata  # Colab Secrets에서 Key 읽기
import openai                      # OpenAI 라이브러리

# Secrets에 저장한 Key로 클라이언트 생성
client = openai.OpenAI(
    api_key=userdata.get("OPENAI_API_KEY")  # 코드에 Key를 직접 쓰지 않습니다
)

print("✅ 클라이언트 생성 완료!")

✅ 클라이언트 생성 완료!


### ⚠️ 오류가 뜬다면?

| 오류 메시지 | 원인 | 해결 방법 |
|------------|------|----------|
| `AuthenticationError` | Key가 잘못됨 | Secrets에서 Key 값 재확인 |
| `SecretNotFoundError` | 이름 불일치 | `OPENAI_API_KEY` 대소문자 확인 |
| `NotFoundError` | 토글 미활성화 | Secrets에서 토글 파란색으로 변경 |

---
## STEP 4. 첫 번째 API 호출 해보기

이제 GPT에게 직접 말을 걸어봅시다!

### 📮 메시지 구조 이해하기

GPT와 대화할 때 `messages` 리스트에 **역할(role)** 과 **내용(content)** 을 담아 보냅니다.

| role | 역할 | 언제 쓰나 |
|------|------|----------|
| `system` | GPT에게 "어떻게 행동할지" 지시 | 맨 처음 한 번 |
| `user` | 나의 질문·요청 | 매번 |
| `assistant` | GPT의 이전 답변 기록 | 대화 이어갈 때 |

In [7]:
# GPT에게 첫 번째 질문을 보냅니다
response = client.chat.completions.create(
    model="gpt-4o-mini",          # 사용할 모델
    messages=[
        {"role": "system", "content": "당신은 친절한 AI 어시스턴트입니다."},
        {"role": "user",   "content": "파인튜닝이 무엇인지 한 문장으로 설명해 주세요."}
    ],
    temperature=1,   # 창의성 수준 (0~2, 높을수록 다양한 답변)
    max_tokens=500,    # 최대 응답 길이
)

# GPT의 답변만 뽑아서 출력
print(response.choices[0].message.content)

파인튜닝은 이미 학습된 모델을 특정 작업이나 데이터에 맞게 추가로 학습시키는 과정을 말합니다.


### 🧪 실험: system 프롬프트가 있을 때 vs 없을 때

같은 질문이라도 system 설정에 따라 답변이 완전히 달라집니다.

In [8]:
# ① system 없이 질문
response_no_sys = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "계약 해지 통보는 언제 해야 하나요?"}
    ],
    temperature=0.3,
    max_tokens=200,
)

# ② system 있이 질문
response_with_sys = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "당신은 대한민국 법률 전문가입니다. 관련 법조항을 인용하여 3문장 이내로 답변하세요."},
        {"role": "user",   "content": "계약 해지 통보는 언제 해야 하나요?"}
    ],
    temperature=0.3,
    max_tokens=200,
)

print("─── system 없음 ───")
print(response_no_sys.choices[0].message.content)

print("\n─── system 있음 ───")
print(response_with_sys.choices[0].message.content)

─── system 없음 ───
계약 해지 통보는 계약서에 명시된 조건과 절차에 따라 진행해야 합니다. 일반적으로 다음과 같은 사항을 고려해야 합니다:

1. **계약서 확인**: 계약서에 해지 통보를 위한 특정 기간이나 절차가 명시되어 있을 수 있습니다. 이를 반드시 확인해야 합니다.

2. **법적 요건**: 특정 계약의 경우, 법적으로 요구되는 해지 통보 기간이 있을 수 있습니다. 예를 들어, 임대차 계약의 경우, 법에서 정한 최소 통보 기간이 있을 수 있습니다.

3. **상대방과의 관계**: 해지 통보를 할 때 상대방과의 관계를 고려하여 적절한 시점에 통보하는 것이 중요합니다. 예를 들어, 상대방이 준비할 시간을 고려해야 할 수 있습니다.

4. **서면 통보**: 가능하면 서면으로 통보하는 것이 좋습니다. 이메일

─── system 있음 ───
계약 해지 통보는 민법 제548조에 따라 계약의 해지를 원하는 당사자는 상대방에게 해지 의사를 명확히 통보해야 하며, 통보 시점은 계약의 위반 사실을 인지한 때부터 가능합니다. 또한, 계약서에 별도의 해지 통보 기한이 명시되어 있다면 그에 따라야 합니다. 따라서, 해지 통보는 위반 사실을 인지한 즉시 또는 계약서에 명시된 기한 내에 해야 합니다.


In [9]:
print(response)

ChatCompletion(id='chatcmpl-DyW7qPl4krjZyFfFMnff53vYhnahM', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='파인튜닝은 이미 학습된 모델을 특정 작업이나 데이터에 맞게 추가로 학습시키는 과정을 말합니다.', refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=None))], created=1783315542, model='gpt-4o-mini-2024-07-18', object='chat.completion', moderation=None, service_tier='default', system_fingerprint='fp_d0469e1700', usage=CompletionUsage(completion_tokens=29, prompt_tokens=40, total_tokens=69, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [ ]:
ChatCompletion(id='chatcmpl-DyW7qPl4krjZyFfFMnff53vYhnahM',
               choices=[Choice(finish_reason='stop',
                               index=0,
                               logprobs=None,
                               message=ChatCompletionMessage(content='파인튜닝은 이미 학습된 모델을 특정 작업이나 데이터에 맞게 추가로 학습시키는 과정을 말합니다.',
                                                             refusal=None,
                                                             role='assistant',
                                                             annotations=[],
                                                             audio=None,
                                                             function_call=None,
                                                             tool_calls=None))],
               created=1783315542,
               model='gpt-4o-mini-2024-07-18',
               object='chat.completion',
               moderation=None,
               service_tier='default',
               system_fingerprint='fp_d0469e1700',
               usage=CompletionUsage(
                   completion_tokens=29,
                   prompt_tokens=40,
                   total_tokens=69,
                   completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0,
                                                                     audio_tokens=0,
                                                                     reasoning_tokens=0,
                                                                     rejected_prediction_tokens=0),
                   prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))



### 🎛️ 주요 파라미터 정리

| 파라미터 | 필수? | 설명 | 예시 |
|---------|-------|------|------|
| `model` | ✅ | 사용할 모델 이름 | `"gpt-4o-mini"` |
| `messages` | ✅ | 대화 내용 리스트 | `[{"role":..., "content":...}]` |
| `temperature` | 선택 | 창의성 (0=일정, 2=다양) | `0.7` |
| `max_tokens` | 선택 | 최대 출력 길이 | `256` |
| `top_p` | 선택 | 단어 선택 범위 조절 | `0.9` |

---
## STEP 5. 응답 구조 파헤치기

GPT의 응답은 단순한 텍스트가 아닙니다. **여러 정보를 담은 객체**입니다.

```
response
├── choices[0].message.content   ← 실제 답변 텍스트 ✅ (가장 많이 씀)
├── choices[0].finish_reason     ← 왜 대답을 멈췄나?
└── usage
    ├── prompt_tokens            ← 입력에 쓴 토큰 수
    ├── completion_tokens        ← 출력에 쓴 토큰 수
    └── total_tokens             ← 전체 토큰 수 (과금 기준)
```

In [10]:
# 앞에서 받은 response 를 해부해봅니다
print("📝 답변 내용:")
print(response.choices[0].message.content)   # 실제 답변

print("\n🏁 종료 이유:", response.choices[0].finish_reason)  # stop / length

print("\n📊 토큰 사용량:")
print("   입력 토큰:", response.usage.prompt_tokens)
print("   출력 토큰:", response.usage.completion_tokens)
print("   전체 토큰:", response.usage.total_tokens)

📝 답변 내용:
파인튜닝은 이미 학습된 모델을 특정 작업이나 데이터에 맞게 추가로 학습시키는 과정을 말합니다.

🏁 종료 이유: stop

📊 토큰 사용량:
   입력 토큰: 40
   출력 토큰: 29
   전체 토큰: 69


### 🚦 finish_reason 이란?

| 값 | 의미 | 대응 방법 |
|----|------|----------|
| `stop` | 정상 종료 ✅ | 그대로 사용 |
| `length` | `max_tokens` 초과로 잘림 ⚠️ | `max_tokens` 값 늘리기 |
| `content_filter` | 안전 정책에 걸림 | 프롬프트 수정 |

> ⚠️ `length` 가 뜨면 답변이 중간에 잘린 겁니다.  
> 특히 JSON 형식 응답이 잘리면 파싱 오류가 납니다!

In [11]:
# 🧪 실험: max_tokens 를 너무 작게 하면 어떻게 될까?
response_short = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "user", "content": "대한민국 헌법의 기본 원리를 상세히 설명해 주세요."}
    ],
    max_tokens=20,   # 의도적으로 아주 짧게 설정
)

print("답변:", response_short.choices[0].message.content)
print("종료 이유:", response_short.choices[0].finish_reason)  # → "length"
print("→ 문장이 중간에 잘렸습니다! max_tokens를 늘려야 합니다.")

답변: 대한민국 헌법의 기본 원리는 국체와 기본이념을 포함하며, 주
종료 이유: length
→ 문장이 중간에 잘렸습니다! max_tokens를 늘려야 합니다.


---
## STEP 6. 비용 계산해보기

### 💰 토큰이란?

GPT가 텍스트를 읽는 **최소 단위**입니다. 과금의 기준이 됩니다.

```
영어: "hello"    → 1 토큰
한국어: "안녕하세요" → 약 3~5 토큰  (한국어가 토큰을 더 씁니다!)
```

### 💵 gpt-4o-mini 단가 (2025년 기준)

| 구분 | 가격 |
|------|------|
| 입력 토큰 | $0.15 / 100만 토큰 |
| 출력 토큰 | $0.60 / 100만 토큰 |

In [12]:
# 방금 호출한 API의 비용을 직접 계산해봅니다
INPUT_PRICE  = 0.15   # 입력: $0.15 / 100만 토큰
OUTPUT_PRICE = 0.60   # 출력: $0.60 / 100만 토큰

# 사용한 토큰 수 가져오기
input_tokens  = response.usage.prompt_tokens
output_tokens = response.usage.completion_tokens

# 비용 계산 (달러)
total_cost = (input_tokens * INPUT_PRICE + output_tokens * OUTPUT_PRICE) / 1_000_000

print(f"입력: {input_tokens}토큰 / 출력: {output_tokens}토큰")
print(f"이번 호출 비용: ${total_cost:.6f}  (약 {total_cost * 1400:.4f}원)")
print(f"1,000번 호출하면: 약 {total_cost * 1000 * 1400:.0f}원")

입력: 40토큰 / 출력: 29토큰
이번 호출 비용: $0.000023  (약 0.0328원)
1,000번 호출하면: 약 33원


### 📊 누적 비용 추적기 만들기

앞으로 API를 많이 호출할 텐데, 비용을 자동으로 합산해주는 도구를 만들어둡니다.  
**Day 2~3에서 그대로 재사용합니다.**

In [13]:
# 호출할 때마다 토큰과 비용을 쌓아주는 추적기
class TokenTracker:
    def __init__(self):
        self.total_input  = 0
        self.total_output = 0
        self.call_count   = 0

    def add(self, response):               # API 응답을 넣으면 자동으로 집계
        self.total_input  += response.usage.prompt_tokens
        self.total_output += response.usage.completion_tokens
        self.call_count   += 1

    def report(self):                      # 지금까지 쓴 비용 출력
        cost = (self.total_input * 0.15 + self.total_output * 0.60) / 1_000_000
        print(f"총 {self.call_count}회 호출 | 입력 {self.total_input} + 출력 {self.total_output} 토큰")
        print(f"누적 비용: ${cost:.4f} (약 {cost * 1400:.0f}원)")

# 사용법 예시
tracker = TokenTracker()
tracker.add(response)     # 아까 받은 응답 등록
tracker.report()          # 지금까지 쓴 비용 출력

총 1회 호출 | 입력 40 + 출력 29 토큰
누적 비용: $0.0000 (약 0원)


---
## ✅ 오늘 실습 체크리스트

| 항목 | 확인 |
|------|------|
| API Key 발급 완료 | ☐ |
| Colab Secrets 저장 + 토글 활성화 | ☐ |
| 클라이언트 생성 성공 (`✅ 클라이언트 생성 완료!`) | ☐ |
| 첫 API 호출 → 응답 텍스트 출력 성공 | ☐ |
| system 유무에 따른 답변 차이 확인 | ☐ |
| `finish_reason: length` 현상 체험 | ☐ |
| 비용 계산 코드 실행 성공 | ☐ |
| `TokenTracker` 만들고 `report()` 출력 성공 | ☐ |

---

## 🔜 다음 시간 예고

> **Day 2** 에서는 GPT를 활용해 **학습 데이터를 자동으로 생성**하는 방법을 배웁니다.  
> 오늘 만든 `client` 와 `TokenTracker` 를 그대로 가져와서 사용합니다!